# 小红书素材包手动定时发布

把下面配置单元里的 `MATERIALS_INPUT` 改成你要发布的素材路径，然后按顺序运行各单元。

当前示例素材包：

`/Users/wangbin/data/jupyter/实战/爬网页/小红书/自动化工作流/xhs每日生成素材/2026-05-09/AI资讯一手信息源`

支持三种输入：

- 单篇素材包目录：`.../2026-05-09/AI资讯一手信息源`
- 某天素材目录：`.../2026-05-09`
- 直接指向 `00_自动发推适配/publish_manifest.json`

发布逻辑：

- 自动读取 `publish_manifest.json`，按 manifest 提取标题、正文、封面和配图路径。
- 标题限制 20 字，正文限制 1000 字，图片限制 18 张。
- 默认从素材日期的第二天开始定时，时间段为 `12:00`、`16:00`；也可以在配置里手动指定精确时间。
- 运行最后一个单元会点击“小红书定时发布”。运行前请确认 9209 端口对应浏览器已登录创作者后台账号。


In [17]:
from __future__ import annotations

import hashlib
import json
import re
from dataclasses import dataclass
from datetime import datetime, timedelta
from pathlib import Path
from time import sleep
from typing import Any

import pandas as pd

ROOT = Path('/Users/wangbin/data/jupyter/实战/爬网页/小红书/自动化工作流')

# 手动运行时只改这里：可以填单篇素材包、日期目录，或 publish_manifest.json。
MATERIALS_INPUT = Path('/Users/wangbin/data/jupyter/实战/爬网页/小红书/自动化工作流/xhs每日生成素材/2026-05-15/情绪复盘卡片_可发布版')
MATERIALS_PATH = MATERIALS_INPUT.expanduser()
if not MATERIALS_PATH.is_absolute():
    MATERIALS_PATH = ROOT / MATERIALS_PATH
MATERIALS_PATH = MATERIALS_PATH.resolve()


def infer_materials_date(path: Path) -> str:
    for part in reversed(path.parts):
        if re.fullmatch(r'\d{4}-\d{2}-\d{2}', part):
            return part
    return datetime.now().strftime('%Y-%m-%d')


MATERIALS_DATE = infer_materials_date(MATERIALS_PATH)

# 9209 是你成功基线 debug.ipynb 里使用的端口。
CHROMIUM_PORT = 9209

# 默认会真正点击“定时发布”。如果只想测试填充页面，把它改成 False。
SUBMIT =  False

# True 时会根据发布历史跳过已经成功提交过的素材；需要重发同一素材时改成 False。
SKIP_ALREADY_PUBLISHED = False

# 如需指定精确发布时间，填满待发布篇数，例如：('2026-05-10 12:00',)。为空则自动生成。
MANUAL_SCHEDULE_TIMES: tuple[str, ...] = ()

# 自动定时时间：素材日期的第二天开始排。如果补跑时日期已过，会自动顺延到运行日的明天。
DAILY_TIME_SLOTS = ('12:00', '16:00')

# 可选：当 MATERIALS_INPUT 是日期目录时，用这里固定发布顺序；为空时按文件夹名排序。
TOPIC_ORDER: list[str] = []

# 发布成功历史：用于后续自动跳过，避免重复定时发布同一素材。
HISTORY_DIR = ROOT / 'outputs' / 'publish_history'
HISTORY_XLSX = HISTORY_DIR / 'published_history.xlsx'
HISTORY_CSV = HISTORY_DIR / 'published_history.csv'
HISTORY_JSONL = HISTORY_DIR / 'published_history.jsonl'

TITLE_LIMIT = 20
BODY_LIMIT = 1000
IMAGE_LIMIT = 18
TOPIC_FIELDS = ('recommended_topics', 'topic_tags', 'topics', 'tags', 'hot_topics')
COLLECTION_FIELDS = ('collection_name', 'collection_title', 'target_collection', 'collection')
COLLECTION_INTRO_FIELDS = ('collection_intro', 'collection_description', 'collection_desc')
COLLECTION_TITLE_LIMIT = 20
COLLECTION_INTRO_LIMIT = 50
TAROT_COLLECTION = '塔罗牌合集'
DATA_COLLECTION = '数据资源的合集'
CASUAL_COLLECTION = '随便发发合集'
FIXED_COLLECTIONS = (TAROT_COLLECTION, DATA_COLLECTION, CASUAL_COLLECTION)
DATA_COLLECTION_KEYWORDS = (
    '数据', '爬虫', '爬取', '采集', '公开', '字段', '表格', '分析',
    '可视化', '清洗', 'python', 'pandas', 'excel', 'api',
)
TAROT_COLLECTION_KEYWORDS = ('塔罗', 'tarot')

print('手动素材输入：', MATERIALS_INPUT)
print('解析后路径：', MATERIALS_PATH)
print('素材日期：', MATERIALS_DATE)
print('浏览器端口：', CHROMIUM_PORT)
print('是否点击定时发布：', SUBMIT)
print('跳过已发布素材：', SKIP_ALREADY_PUBLISHED)
print('发布历史表：', HISTORY_XLSX)


手动素材输入： /Users/wangbin/data/jupyter/实战/爬网页/小红书/自动化工作流/xhs每日生成素材/2026-05-15/情绪复盘卡片_可发布版
解析后路径： /Users/wangbin/data/jupyter/实战/爬网页/小红书/自动化工作流/outputs/materials/2026-05-15/情绪复盘卡片_可发布版
素材日期： 2026-05-15
浏览器端口： 9209
是否点击定时发布： False
跳过已发布素材： False
发布历史表： /Users/wangbin/data/jupyter/实战/爬网页/小红书/自动化工作流/outputs/publish_history/published_history.xlsx


In [18]:
from __future__ import annotations

@dataclass
class PublishItem:
    topic_folder: str
    manifest_path: Path
    publish_key: str
    title: str
    body: str
    images: list[Path]
    topic_tags: list[str]
    collection_name: str
    collection_intro: str
    schedule_time: str | None = None


HISTORY_COLUMNS = [
    'recorded_at',
    'status',
    'publish_key',
    'topic_folder',
    'title',
    'body_chars',
    'image_count',
    'topic_tags_json',
    'collection_name',
    'schedule_time',
    'manifest_path',
    'cover_path',
    'image_paths_json',
    'materials_date',
    'message',
]
SUCCESS_STATUSES = {'scheduled', 'scheduled_click_success', 'published', 'submitted', 'success'}


def package_dir_for_manifest(manifest_path: Path) -> Path:
    return manifest_path.parents[1]


def resolve_image_path(raw_path: str, manifest_path: Path) -> Path:
    raw_path = str(raw_path).strip()
    path = Path(raw_path).expanduser()
    package_dir = package_dir_for_manifest(manifest_path)

    candidates: list[Path] = []
    if path.is_absolute():
        candidates.append(path)
    else:
        candidates.extend([package_dir / path, manifest_path.parent / path, ROOT / path])

    daily_root = ROOT / 'xhs每日生成素材'
    output_root = ROOT / 'outputs' / 'materials'
    root_pairs = ((daily_root, output_root), (output_root, daily_root))
    for candidate in list(candidates):
        for src_root, dst_root in root_pairs:
            try:
                candidates.append(dst_root / candidate.relative_to(src_root))
            except ValueError:
                pass

    for candidate in candidates:
        if candidate.exists():
            return candidate
    return candidates[0] if candidates else path


def load_manifest(path: Path) -> dict[str, Any]:
    with path.open('r', encoding='utf-8') as f:
        data = json.load(f)
    if not isinstance(data, dict):
        raise ValueError(f'manifest 格式不正确：{path}')
    return data


def publish_key_for_manifest(manifest_path: Path, manifest: dict[str, Any]) -> str:
    row = manifest.get('publish_row') or {}
    image_paths = [str(item.get('path', '')) for item in sorted(manifest.get('images', []), key=lambda item: item.get('upload_order', 999))]
    payload = {
        'topic_folder': package_dir_for_manifest(manifest_path).name,
        'title': str(manifest.get('recommended_title') or row.get('标题') or '').strip(),
        'body': str(manifest.get('body') or row.get('推文') or '').strip(),
        'images': image_paths,
    }
    text = json.dumps(payload, ensure_ascii=False, sort_keys=True)
    return hashlib.sha256(text.encode('utf-8')).hexdigest()


def pick_text(manifest: dict[str, Any]) -> tuple[str, str]:
    row = manifest.get('publish_row') or {}
    title = str(manifest.get('recommended_title') or row.get('标题') or '').strip()
    body = str(manifest.get('body') or row.get('推文') or '').strip()
    return title[:TITLE_LIMIT], body[:BODY_LIMIT]



def normalize_topic_tag(raw: Any) -> str:
    return str(raw or '').strip().lstrip('#').strip()


def pick_topic_tags(manifest: dict[str, Any]) -> list[str]:
    tags: list[str] = []
    for field in TOPIC_FIELDS:
        raw = manifest.get(field)
        if raw is None:
            continue
        if isinstance(raw, str):
            candidates = re.split(r'[\s,，、;；|]+', raw)
        elif isinstance(raw, list):
            candidates = raw
        else:
            candidates = [raw]
        for candidate in candidates:
            tag = normalize_topic_tag(candidate)
            if tag and tag not in tags:
                tags.append(tag)
        if tags:
            break
    return tags



def _first_manifest_text(manifest: dict[str, Any], fields: tuple[str, ...]) -> str:
    for field in fields:
        raw = manifest.get(field)
        if isinstance(raw, dict):
            raw = raw.get('name') or raw.get('title')
        text = str(raw or '').strip()
        if text:
            return text
    return ''


def _short_text(value: str, limit: int) -> str:
    return str(value or '').strip()[:limit]


def canonical_collection_name(text: str) -> str | None:
    normalized = re.sub(r'\s+', '', str(text or '').strip()).lower()
    if not normalized:
        return None
    for collection in FIXED_COLLECTIONS:
        collection_norm = re.sub(r'\s+', '', collection).lower()
        if collection_norm in normalized or normalized in collection_norm:
            return collection
    if any(keyword.lower() in normalized for keyword in TAROT_COLLECTION_KEYWORDS):
        return TAROT_COLLECTION
    if any(keyword.lower() in normalized for keyword in DATA_COLLECTION_KEYWORDS):
        return DATA_COLLECTION
    return None


def infer_collection_name(manifest: dict[str, Any], title: str, topic_folder: str, topic_tags: list[str]) -> str:
    explicit = _first_manifest_text(manifest, COLLECTION_FIELDS)
    if explicit:
        canonical = canonical_collection_name(explicit)
        if canonical:
            return canonical

    text = ' '.join([title, topic_folder, ' '.join(topic_tags)])
    canonical = canonical_collection_name(text)
    return canonical or CASUAL_COLLECTION


def infer_collection_intro(manifest: dict[str, Any], collection_name: str, title: str, topic_tags: list[str]) -> str:
    explicit = _first_manifest_text(manifest, COLLECTION_INTRO_FIELDS)
    if explicit:
        return _short_text(explicit, COLLECTION_INTRO_LIMIT)
    if topic_tags:
        intro = '、'.join(topic_tags[:4]) + '相关内容整理'
    else:
        intro = f'{title}相关内容整理'
    return _short_text(intro or f'{collection_name}简介', COLLECTION_INTRO_LIMIT)


def pick_collection(manifest: dict[str, Any], title: str, topic_folder: str, topic_tags: list[str]) -> tuple[str, str]:
    collection_name = infer_collection_name(manifest, title, topic_folder, topic_tags)
    collection_intro = infer_collection_intro(manifest, collection_name, title, topic_tags)
    return collection_name, collection_intro


def ordered_images(manifest: dict[str, Any], manifest_path: Path) -> list[Path]:
    items = sorted(manifest.get('images', []), key=lambda item: item.get('upload_order', 999))
    return [resolve_image_path(str(item.get('path', '')), manifest_path) for item in items]


def discover_manifests(input_path: Path) -> list[Path]:
    path = input_path.expanduser()
    if path.is_file():
        if path.name != 'publish_manifest.json':
            raise ValueError(f'文件输入必须是 publish_manifest.json：{path}')
        return [path]

    direct_manifest = path / '00_自动发推适配' / 'publish_manifest.json'
    if direct_manifest.exists():
        return [direct_manifest]

    adapter_manifest = path / 'publish_manifest.json'
    if path.name == '00_自动发推适配' and adapter_manifest.exists():
        return [adapter_manifest]

    manifests = list(path.glob('*/00_自动发推适配/publish_manifest.json'))
    if TOPIC_ORDER:
        order = {name: idx for idx, name in enumerate(TOPIC_ORDER)}
        manifests.sort(key=lambda p: (order.get(package_dir_for_manifest(p).name, 999), package_dir_for_manifest(p).name))
    else:
        manifests.sort(key=lambda p: package_dir_for_manifest(p).name)
    return manifests


def normalize_schedule_time(value: str) -> str:
    try:
        publish_at = datetime.strptime(value, '%Y-%m-%d %H:%M')
    except ValueError as exc:
        raise ValueError(f'定时时间格式应为 YYYY-MM-DD HH:MM：{value}') from exc
    if publish_at <= datetime.now():
        raise ValueError(f'定时时间必须晚于当前时间：{value}')
    return publish_at.strftime('%Y-%m-%d %H:%M')


def generate_schedule_times(count: int) -> list[str]:
    if MANUAL_SCHEDULE_TIMES:
        if len(MANUAL_SCHEDULE_TIMES) < count:
            raise ValueError(f'MANUAL_SCHEDULE_TIMES 只有 {len(MANUAL_SCHEDULE_TIMES)} 个，但待发布素材有 {count} 篇。')
        return [normalize_schedule_time(value) for value in MANUAL_SCHEDULE_TIMES[:count]]

    material_date = datetime.strptime(MATERIALS_DATE, '%Y-%m-%d').date()
    start_date = material_date + timedelta(days=1)

    # 如果以后补跑，避免生成已经过去的时间，自动顺延到运行日的明天。
    tomorrow = datetime.now().date() + timedelta(days=1)
    if start_date < tomorrow:
        start_date = tomorrow

    slots: list[str] = []
    day = start_date
    while len(slots) < count:
        for slot in DAILY_TIME_SLOTS:
            slots.append(normalize_schedule_time(f'{day:%Y-%m-%d} {slot}'))
            if len(slots) >= count:
                break
        day += timedelta(days=1)
    return slots


def validate_item(item: PublishItem) -> list[str]:
    issues: list[str] = []
    if not item.title:
        issues.append('缺少标题')
    if len(item.title) > TITLE_LIMIT:
        issues.append(f'标题超过 {TITLE_LIMIT} 字：{len(item.title)}')
    if not item.body:
        issues.append('缺少正文')
    if len(item.body) > BODY_LIMIT:
        issues.append(f'正文超过 {BODY_LIMIT} 字：{len(item.body)}')
    if not item.topic_tags:
        issues.append('缺少话题标签：请在 manifest 中提供 recommended_topics/topic_tags/topics/tags')
    if not item.collection_name:
        issues.append('缺少合集名称')
    if len(item.collection_name) > COLLECTION_TITLE_LIMIT:
        issues.append(f'合集名称超过 {COLLECTION_TITLE_LIMIT} 字：{len(item.collection_name)}')
    if len(item.collection_intro) > COLLECTION_INTRO_LIMIT:
        issues.append(f'合集简介超过 {COLLECTION_INTRO_LIMIT} 字：{len(item.collection_intro)}')
    if not item.images:
        issues.append('缺少图片')
    if len(item.images) > IMAGE_LIMIT:
        issues.append(f'图片超过 {IMAGE_LIMIT} 张：{len(item.images)}')
    if item.schedule_time:
        try:
            normalize_schedule_time(item.schedule_time)
        except ValueError as exc:
            issues.append(str(exc))
    for image in item.images:
        if not image.exists():
            issues.append(f'图片不存在：{image}')
    return issues


def load_publish_history() -> pd.DataFrame:
    if HISTORY_XLSX.exists():
        df = pd.read_excel(HISTORY_XLSX, dtype=str).fillna('')
    elif HISTORY_CSV.exists():
        df = pd.read_csv(HISTORY_CSV, dtype=str).fillna('')
    else:
        df = pd.DataFrame(columns=HISTORY_COLUMNS)
    for col in HISTORY_COLUMNS:
        if col not in df.columns:
            df[col] = ''
    return df[HISTORY_COLUMNS]


def load_published_keys() -> set[str]:
    if not SKIP_ALREADY_PUBLISHED:
        return set()
    history = load_publish_history()
    if history.empty:
        return set()
    status = history['status'].astype(str).str.lower()
    return set(history.loc[status.isin(SUCCESS_STATUSES), 'publish_key'].astype(str))


def make_history_record(item: PublishItem, *, status: str, message: str = '') -> dict[str, Any]:
    return {
        'recorded_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'status': status,
        'publish_key': item.publish_key,
        'topic_folder': item.topic_folder,
        'title': item.title,
        'body_chars': len(item.body),
        'image_count': len(item.images),
        'topic_tags_json': json.dumps(item.topic_tags, ensure_ascii=False),
        'collection_name': item.collection_name,
        'schedule_time': item.schedule_time or '',
        'manifest_path': str(item.manifest_path),
        'cover_path': str(item.images[0]) if item.images else '',
        'image_paths_json': json.dumps([str(p) for p in item.images], ensure_ascii=False),
        'materials_date': MATERIALS_DATE,
        'message': message,
    }


def append_publish_history(item: PublishItem, *, status: str = 'scheduled_click_success', message: str = '已点击定时发布') -> Path:
    HISTORY_DIR.mkdir(parents=True, exist_ok=True)
    record = make_history_record(item, status=status, message=message)
    history = load_publish_history()
    history = pd.concat([history, pd.DataFrame([record])], ignore_index=True)
    history.to_excel(HISTORY_XLSX, index=False)
    history.to_csv(HISTORY_CSV, index=False, encoding='utf-8-sig')
    with HISTORY_JSONL.open('a', encoding='utf-8') as f:
        f.write(json.dumps(record, ensure_ascii=False) + '\n')
    return HISTORY_XLSX


def build_completion_body(processed_items: list[PublishItem], skipped_count: int = 0) -> str:
    lines = [
        f'时间：{datetime.now().strftime("%Y-%m-%d %H:%M:%S")}',
        f'素材输入：{MATERIALS_PATH}',
        f'素材日期：{MATERIALS_DATE}',
        f'本次检测发布成功：{len(processed_items)} 篇',
        f'因历史记录跳过：{skipped_count} 篇',
        f'发布历史表：{HISTORY_XLSX}',
        '',
    ]
    if processed_items:
        lines.append('本次发布明细：')
        for idx, item in enumerate(processed_items, 1):
            lines.append(f'{idx}. {item.topic_folder}｜{item.title}｜{item.schedule_time}｜图片{len(item.images)}张')
    else:
        lines.append('本次没有新的待发布素材。')
    return '\n'.join(lines)


def print_publish_completion_report(processed_items: list[PublishItem], skipped_count: int = 0) -> None:
    print('发布完成汇总：')
    print(build_completion_body(processed_items, skipped_count=skipped_count))


def load_publish_queue() -> list[PublishItem]:
    manifests = discover_manifests(MATERIALS_PATH)
    if not manifests:
        raise FileNotFoundError(f'没有找到 publish_manifest.json：{MATERIALS_PATH}')

    print(f'找到 {len(manifests)} 个待检查 manifest。')
    published_keys = load_published_keys()
    pending: list[tuple[Path, dict[str, Any], str]] = []
    skipped: list[tuple[str, str]] = []
    for manifest_path in manifests:
        manifest = load_manifest(manifest_path)
        publish_key = publish_key_for_manifest(manifest_path, manifest)
        if publish_key in published_keys:
            skipped.append((package_dir_for_manifest(manifest_path).name, publish_key[:12]))
            continue
        pending.append((manifest_path, manifest, publish_key))

    if skipped:
        print(f'根据发布历史跳过 {len(skipped)} 篇已发布素材：')
        for topic, key in skipped:
            print(f'  - {topic} ({key})')

    schedule_times = generate_schedule_times(len(pending))
    queue: list[PublishItem] = []
    for (manifest_path, manifest, publish_key), schedule_time in zip(pending, schedule_times):
        title, body = pick_text(manifest)
        topic_tags = pick_topic_tags(manifest)
        collection_name, collection_intro = pick_collection(manifest, title, package_dir_for_manifest(manifest_path).name, topic_tags)
        images = ordered_images(manifest, manifest_path)
        queue.append(
            PublishItem(
                topic_folder=package_dir_for_manifest(manifest_path).name,
                manifest_path=manifest_path,
                publish_key=publish_key,
                title=title,
                body=body,
                images=images,
                topic_tags=topic_tags,
                collection_name=collection_name,
                collection_intro=collection_intro,
                schedule_time=schedule_time,
            )
        )
    return queue


In [19]:
publish_queue = load_publish_queue()

for idx, item in enumerate(publish_queue, 1):
    issues = validate_item(item)
    print(f'[{idx}] {item.topic_folder}')
    print('  标题：', item.title, f'({len(item.title)}/{TITLE_LIMIT})')
    print('  正文：', f'{len(item.body)}/{BODY_LIMIT}')
    print('  图片：', len(item.images), f'/{IMAGE_LIMIT}')
    print('  话题：', '、'.join(item.topic_tags) if item.topic_tags else '未设置')
    print('  合集：', item.collection_name, f'({len(item.collection_name)}/{COLLECTION_TITLE_LIMIT})')
    print('  定时：', item.schedule_time)
    print('  manifest：', item.manifest_path)
    for image in item.images:
        print('   -', image)
    if issues:
        print('  校验问题：')
        for issue in issues:
            print('   !', issue)
    print()

all_issues = [(item.topic_folder, validate_item(item)) for item in publish_queue]
all_issues = [(topic, issues) for topic, issues in all_issues if issues]
if all_issues:
    raise ValueError('发布校验未通过，请先处理上面列出的问题。')

print('发布队列校验通过。最后一个单元会按上面的定时时间提交。')


找到 1 个待检查 manifest。
[1] 情绪复盘卡片_可发布版
  标题： 情绪复盘先写卡 (7/20)
  正文： 287/1000
  图片： 6 /18
  话题： 情绪复盘、自我提问、沟通表达、关系沟通、生活复盘
  合集： 随便发发合集 (6/20)
  定时： 2026-05-17 12:00
  manifest： /Users/wangbin/data/jupyter/实战/爬网页/小红书/自动化工作流/outputs/materials/2026-05-15/情绪复盘卡片_可发布版/00_自动发推适配/publish_manifest.json
   - /Users/wangbin/data/jupyter/实战/爬网页/小红书/自动化工作流/outputs/materials/2026-05-15/情绪复盘卡片_可发布版/03_配图/压缩上传版/page_01_upload.jpg
   - /Users/wangbin/data/jupyter/实战/爬网页/小红书/自动化工作流/outputs/materials/2026-05-15/情绪复盘卡片_可发布版/03_配图/压缩上传版/page_02_upload.jpg
   - /Users/wangbin/data/jupyter/实战/爬网页/小红书/自动化工作流/outputs/materials/2026-05-15/情绪复盘卡片_可发布版/03_配图/压缩上传版/page_03_upload.jpg
   - /Users/wangbin/data/jupyter/实战/爬网页/小红书/自动化工作流/outputs/materials/2026-05-15/情绪复盘卡片_可发布版/03_配图/压缩上传版/page_04_upload.jpg
   - /Users/wangbin/data/jupyter/实战/爬网页/小红书/自动化工作流/outputs/materials/2026-05-15/情绪复盘卡片_可发布版/03_配图/压缩上传版/page_05_upload.jpg
   - /Users/wangbin/data/jupyter/实战/爬网页/小红书/自动化工作流/outputs/materials/2026-05-15/情绪复盘卡片_可发布版/03_配

In [20]:
from __future__ import annotations

def connect_browser(port: int = CHROMIUM_PORT):
    from DrissionPage import ChromiumOptions, ChromiumPage
    from DrissionPage.common import Actions

    co = (
        ChromiumOptions()
        .set_paths(local_port=port)
        .mute(True)
        .set_argument('--start-maximized')
        .ignore_certificate_errors(True)
    )
    page = ChromiumPage(co)
    page.set.auto_handle_alert()
    actions = Actions(page)
    page.get('https://creator.xiaohongshu.com/publish/publish?from=menu')
    return page, actions


def open_image_publish_page(page, actions) -> None:
    try:
        page.get('https://creator.xiaohongshu.com/publish/publish?from=menu&target=image')
    except Exception:
        page.get('https://creator.xiaohongshu.com/publish/publish')
    sleep(2)
    try:
        tab = page.ele('text=上传图文', timeout=2)
        actions.move_to(ele_or_loc=tab).click()
    except Exception:
        pass


def scroll_down(page, actions, distance: int = 700) -> None:
    try:
        size = page.run_js('return {w: window.innerWidth, h: window.innerHeight};')
        actions.move_to((size['w'] // 2, size['h'] // 2))
    except Exception:
        pass

    total = 0
    while total < distance:
        actions.scroll(delta_y=70)
        total += 70
        sleep(0.08)


def human_scroll_down(page, actions, distance: int) -> None:
    from random import randint, uniform

    sleep(3)
    try:
        size = page.run_js('return {w: window.innerWidth, h: window.innerHeight};')
        center_x = size['w'] // 2
        center_y = size['h'] // 2
        actions.move_to((center_x, center_y))
    except Exception:
        pass

    total = 0
    while total < distance:
        step = randint(35, 70)
        actions.scroll(delta_y=step)
        total += step
        sleep(uniform(0.08, 0.18))


def find_upload_control(page, first_image: bool = False):
    selectors = ('tx=上传图片', '.upload-wrapper', '.entry') if first_image else ('.entry', 'tx=上传图片', '.upload-wrapper')
    for selector in selectors:
        try:
            ele = page.ele(selector, timeout=5)
            if selector == '.upload-wrapper':
                try:
                    return ele.ele('.upload-input', timeout=1)
                except Exception:
                    return ele
            return ele
        except Exception:
            continue
    raise RuntimeError('找不到上传图片控件。')


def upload_images(page, images: list[Path]) -> None:
    for idx, image in enumerate(images):
        upload = find_upload_control(page, first_image=(idx == 0))
        upload.click.to_upload(str(image))
        page.wait(2, 3) if idx == 0 else page.wait(1, 2)


def fill_field(page, selectors: tuple[str, ...], value: str, field_name: str):
    last_error: Exception | None = None
    for selector in selectors:
        try:
            ele = page.ele(selector, timeout=4)
            ele.input(value)
            return ele
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f'找不到{field_name}输入框。') from last_error


def move_body_cursor_after_input(actions) -> None:
    from DrissionPage.common import Keys

    for _ in range(2):
        actions.key_down(Keys.DOWN).key_up(Keys.DOWN)
        sleep(0.05)
    actions.key_down(Keys.END).key_up(Keys.END)


def input_preselected_topic_tags(input_target, topic_tags: list[str]) -> None:
    for tag in topic_tags:
        input_target.input(f'#{tag}')
        sleep(0.5)
        input_target.input('\n')



def normalize_collection_text(value: str) -> str:
    return re.sub(r'\s+', '', str(value or '').strip())


def parse_collection_names(raw_text: str) -> list[str]:
    names: list[str] = []
    for line in str(raw_text or '').splitlines():
        name = line.strip()
        if not name or '创建合集' in name or name in {'合集', '选择合集'}:
            continue
        if name not in names:
            names.append(name)
    return names


def collection_match_score(target: str, existing: str, topic_tags: list[str]) -> int:
    target_norm = normalize_collection_text(target)
    existing_norm = normalize_collection_text(existing)
    if not target_norm or not existing_norm:
        return 0
    if target_norm == existing_norm:
        return 100
    if target_norm in existing_norm or existing_norm in target_norm:
        return 80
    score = 0
    for tag in topic_tags:
        tag_norm = normalize_collection_text(tag)
        if tag_norm and (tag_norm in existing_norm or existing_norm in tag_norm):
            score += 20
    for keyword in ('数据', '爬虫', '采集', '公开', '分析', '塔罗', '旅行', '拍照', '国风', '搞笑'):
        if keyword in target_norm and keyword in existing_norm:
            score += 15
    return score


def best_existing_collection(target: str, existing_names: list[str], topic_tags: list[str]) -> str | None:
    target_norm = normalize_collection_text(target)
    for name in existing_names:
        if normalize_collection_text(name) == target_norm:
            return name
    for name in existing_names:
        existing_norm = normalize_collection_text(name)
        if target_norm and (target_norm in existing_norm or existing_norm in target_norm):
            return name

    scored = [(collection_match_score(target, name, topic_tags), name) for name in existing_names]
    scored = [(score, name) for score, name in scored if score >= 20]
    if not scored:
        return None
    scored.sort(key=lambda item: (-item[0], len(item[1])))
    return scored[0][1]


def open_collection_popover(page, actions):
    sleep(3)
    wrapper = page.ele('.collection-plugin-wrapper', timeout=5)
    actions.move_to(ele_or_loc=wrapper).click()
    sleep(1)
    return page.ele('.collection-plugin-popover-content', timeout=5)


def select_or_create_collection(page, actions, item: PublishItem) -> None:
    popover = open_collection_popover(page, actions)
    existing_text = str(getattr(popover, 'text', '') or '')
    print(existing_text)
    existing_names = parse_collection_names(existing_text)
    target_collection = best_existing_collection(item.collection_name, existing_names, item.topic_tags)

    if target_collection:
        target = popover.ele(f'tx:{target_collection}', timeout=3)
        actions.move_to(ele_or_loc=target).click()
        sleep(1)
        selected = page.ele('.collection-name', timeout=5).text
        print(selected)
        if normalize_collection_text(target_collection) not in normalize_collection_text(selected):
            raise RuntimeError(f'合集选择后校验不一致：期望 {target_collection}，页面显示 {selected}')
        return

    available = '、'.join(existing_names) if existing_names else '未读取到合集'
    fixed = '、'.join(FIXED_COLLECTIONS)
    raise RuntimeError(
        f'未找到目标合集：{item.collection_name}。当前规则只使用已创建的 3 个合集（{fixed}），'
        f'不会自动新建合集。页面已读取合集：{available}'
    )


def declare_original(page, actions) -> None:
    human_scroll_down(page, actions, 600)

    sleep(3)
    try:
        yuanchuang = page.ele('.d-switch-indicator', index=1, timeout=5)
        actions.move_to(ele_or_loc=yuanchuang).click()

        yuanchuang2 = page.ele('.:d-checkbox-indicator', index=1, timeout=5)
        actions.move_to(ele_or_loc=yuanchuang2).click()

        yuanchuang3 = page.ele('tx= 声明原创 ', index=1, timeout=5)
        actions.move_to(ele_or_loc=yuanchuang3).click()
    except Exception as exc:
        raise RuntimeError('声明原创点击失败。') from exc

    human_scroll_down(page, actions, 300)


def set_schedule_time(page, actions, schedule_time: str) -> None:
    from DrissionPage.common import Keys

    scroll_down(page, actions, 700)

    last_error: Exception | None = None
    for selector in ('.custom-switch-switch', '.d-switch-box', 'text=定时发布'):
        try:
            switch = page.ele(selector, index=-1, timeout=4)
            try:
                switch = switch.ele('.d-switch-box', timeout=1)
            except Exception:
                pass
            actions.move_to(ele_or_loc=switch).click()
            last_error = None
            break
        except Exception as exc:
            last_error = exc
    if last_error is not None:
        raise RuntimeError('找不到定时发布开关。') from last_error

    sleep(3)

    input_selectors = (
        '.d-datepicker-suffix --color-text-description d-datepicker-suffix-indicator',
        '.d-datepicker input',
        '@placeholder=请选择时间',
        '@placeholder:请选择时间',
    )
    last_error = None
    for selector in input_selectors:
        try:
            date_input = page.ele(selector, timeout=4)
            actions.move_to(ele_or_loc=date_input).click()
            actions.key_down(Keys.SHIFT)
            for _ in range(24):
                actions.key_down(Keys.LEFT).key_up(Keys.LEFT)
                sleep(0.03)
            actions.key_up(Keys.SHIFT)
            date_input.input(schedule_time)
            sleep(1)
            return
        except Exception as exc:
            last_error = exc
            try:
                actions.key_up(Keys.SHIFT)
            except Exception:
                pass
    raise RuntimeError('找不到定时发布时间输入框。') from last_error


def click_submit(page, actions, schedule_time: str) -> None:
    try:
        host = page.ele('t:xhs-publish-btn', timeout=5)
        button = host.sr.ele('tx=定时发布', timeout=5)
    except Exception:
        button = page.ele('.:publishBtn', timeout=5)
    actions.move_to(ele_or_loc=button).click()
    print(f'已点击定时发布：{schedule_time}')


def check_publish_success(page) -> bool:
    try:
        fabuchengg = page.ele('text=发布成功').text
        print(fabuchengg)
        return True
    except Exception:
        return False


def publish_one_item(page, actions, item: PublishItem, submit: bool = SUBMIT) -> bool:
    issues = validate_item(item)
    if issues:
        raise ValueError(f'{item.topic_folder} 校验失败：' + '；'.join(issues))
    if not item.schedule_time:
        raise ValueError(f'{item.topic_folder} 缺少定时时间。')

    print(f'开始处理：{item.topic_folder} -> {item.schedule_time}')
    open_image_publish_page(page, actions)
    upload_images(page, item.images)

    fill_field(page, ('@placeholder:填写标题', '@placeholder=填写标题会有更多赞哦～'), item.title, '标题')
    body_field = fill_field(
        page,
        ('@data-placeholder:输入正文描述', '@data-placeholder=输入正文描述，真诚有价值的分享予人温暖'),
        item.body,
        '正文',
    )
    body_field.input('\n')
    move_body_cursor_after_input(actions)
    sleep(1)

    input_preselected_topic_tags(body_field, item.topic_tags)
    declare_original(page, actions)
    select_or_create_collection(page, actions, item)
    set_schedule_time(page, actions, item.schedule_time)

    if submit:
        click_submit(page, actions, item.schedule_time)
        page.wait(2, 3)
        return check_publish_success(page)
    else:
        print(f'DRY RUN：已填充并设置定时时间，但未点击发布：{item.schedule_time}')

    page.wait(2, 3)
    return False


## 手动运行发布

运行下面这个单元前，请确认：

1. 配置单元里的 `MATERIALS_INPUT` 已经改成你要发布的素材路径。
2. 预览单元里的标题、正文、图片路径和定时时间都正确。
3. 浏览器端口 `9209` 对应的 Chrome/Chromium 已经登录小红书创作者后台。
4. `SUBMIT = True` 时会真正点击“定时发布”；只想演练填充页面时先改成 `False`。


In [21]:
publish_queue = load_publish_queue()
processed_items: list[PublishItem] = []
initial_manifest_count = len(discover_manifests(MATERIALS_PATH))
skipped_count = max(0, initial_manifest_count - len(publish_queue))

if not publish_queue:
    print('没有新的待发布素材；已根据发布历史全部跳过。')
    print_publish_completion_report(processed_items, skipped_count=skipped_count)
else:
    page, actions = connect_browser(CHROMIUM_PORT)
    try:
        for item in publish_queue:
            publish_success = publish_one_item(page, actions, item, submit=SUBMIT)
            if SUBMIT:
                if not publish_success:
                    raise RuntimeError(f'{item.topic_folder} 未检测到发布成功提示。')
                append_publish_history(item, status='success', message='检测到发布成功')
                print(f'已记录发布历史：{item.topic_folder} -> {HISTORY_XLSX}')
            else:
                print(f'DRY RUN：不写入发布历史：{item.topic_folder}')
            processed_items.append(item)
        print('全部素材处理完成。')
        print_publish_completion_report(processed_items, skipped_count=skipped_count)
    except Exception as exc:
        print(f'发布流程失败：{type(exc).__name__}: {exc}')
        print(f'已成功记录 {len(processed_items)} 篇；剩余 {len(publish_queue) - len(processed_items)} 篇未完成。')
        raise
    finally:
        # 保留浏览器窗口，方便你人工检查发布状态。
        pass


找到 1 个待检查 manifest。
开始处理：情绪复盘卡片_可发布版 -> 2026-05-17 12:00
随便发发
塔罗牌合集
数据资源类型
创建合集
随便发发
DRY RUN：已填充并设置定时时间，但未点击发布：2026-05-17 12:00
DRY RUN：不写入发布历史：情绪复盘卡片_可发布版
全部素材处理完成。
发布完成汇总：
时间：2026-05-16 11:16:01
素材输入：/Users/wangbin/data/jupyter/实战/爬网页/小红书/自动化工作流/outputs/materials/2026-05-15/情绪复盘卡片_可发布版
素材日期：2026-05-15
本次检测发布成功：1 篇
因历史记录跳过：0 篇
发布历史表：/Users/wangbin/data/jupyter/实战/爬网页/小红书/自动化工作流/outputs/publish_history/published_history.xlsx

本次发布明细：
1. 情绪复盘卡片_可发布版｜情绪复盘先写卡｜2026-05-17 12:00｜图片6张


In [22]:
sleep(2)
print(page.url)
button = page.ele('t:xhs-publish-btn', timeout=5).sr.ele('tx=定时发布', index=-1, timeout=5)
actions.move_to(ele_or_loc=button).click()
print(f'已点击定时发布')


https://creator.xiaohongshu.com/publish/publish?from=menu&target=image


ElementNotFoundError: 
没有找到元素。
method: ele()
args: {'locator': 'tx=定时发布', 'index': -1}